# Data Engineer Questions
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Data Engineering, Databases, SQL · **Difficulty/Frequency:** Common (5/10)

*A multi-part screen: 3 SQL problems, 1 data-modeling question, 2 Python coding tasks. Unlike the single-problem notebooks elsewhere in this folder, each part here gets its own concept + idea + runnable check, since they're six genuinely different questions rather than one problem with several approaches.*


## Concepts

**What this screen is really testing (across all parts):**
- SQL window functions (`LAG`/`LEAD`) — comparing a row to its neighbors without a self-join
- `GROUP BY … HAVING` — for finding duplicates / filtering on an aggregate
- Star-schema data modeling — one fact table, several dimension tables
- Two small, self-contained Python data-processing tasks

**Why a window function beats a self-join here:**
- A triple self-join (`s1.id = s2.id - 1 AND s2.id = s3.id - 1`) recomputes the same "who's my neighbor" relationship separately for every candidate triple — that's O(n³) in the naive form.
- A window function computes "what's the value one/two rows away" **once per row, in a single pass** — the database can do this in O(n log n) (for the sort behind `ORDER BY`) instead of a multi-way join.

---

### Quick primers — the building blocks used below

**SQL window functions (`LAG`, `LEAD`).**
- A window function computes a value across a set of *related* rows — without collapsing them into one row the way `GROUP BY` does.
- `LAG(col, k) OVER (ORDER BY x)` returns `col`'s value from k rows **before** the current one (in that order). `LEAD` looks k rows **ahead**.
- Both let you compare a row to its neighbors in a single pass over sorted data — instead of a self-join, which the database effectively has to treat as comparing every pair of rows.

**`GROUP BY` + `HAVING`, for filtering on an aggregate.**
- `GROUP BY` collapses rows that share a key into one row per key, computing an aggregate (`COUNT`, `SUM`, ...) per group.
- `HAVING` filters on that *aggregate*, after grouping — the same way `WHERE` filters individual rows, before grouping.
- `HAVING COUNT(*) > 1` means "keep only groups with more than one row" — i.e., duplicates.

**Star schema (fact table + dimension tables).**
- A **fact table** stores one row per business event, at its finest level of detail (here: one row per sale line item), plus **measures** — numeric values meant to be aggregated, like `sale_amt`.
- **Dimension tables** store the descriptive details of the things the fact table refers to (customer, product, store, date), joined in via foreign keys.
- This shape is built for exactly the query pattern "aggregate a measure, grouped by one or more dimensions" — which is what all three example business questions ask for.

**Why this notebook runs SQL for real.**
- Every query below runs against an in-memory SQLite database (`sqlite3`, part of the Python standard library — no install needed), loaded with the example data.
- That means the outputs are actually checked, not just claimed to look right on paper.
- `DATE_TRUNC` (PostgreSQL) and `DATE_FORMAT` (MySQL) don't exist in SQLite — the MAU query below uses SQLite's own `strftime('%Y-%m', date)` instead, with the PostgreSQL/MySQL/SQL Server equivalents given as comments for interview purposes.


## Part 1 -- SQL: Stadium Consecutive IDs

**Problem:** `Stadium(id, visit_date, people)`, `id` increasing with `visit_date`. Return rows that are part of a run of **3+ consecutive ids**, each with `people >= 100`, ordered by `visit_date`.

**Idea:** a row qualifies if it's part of *any* run of >= 3 consecutive qualifying rows containing it. Using `LAG`/`LEAD` up to offset 2, a row with `people >= 100` qualifies if: (a) it and the **two rows before** it all qualify (it ends a run), OR (b) it has **one qualifying neighbor on each side** (it's in the middle of a run), OR (c) it and the **two rows after** it all qualify (it starts a run). Every row in any run of length >= 3 matches at least one of these three conditions.


In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("""
    CREATE TABLE Stadium (id INTEGER PRIMARY KEY, visit_date TEXT, people INTEGER)
""")
stadium_rows = [
    (1, "2017-01-01", 10), (2, "2017-01-02", 109), (3, "2017-01-03", 150),
    (4, "2017-01-04", 99), (5, "2017-01-05", 145), (6, "2017-01-06", 1455),
    (7, "2017-01-07", 199), (8, "2017-01-09", 188),
]
conn.executemany("INSERT INTO Stadium VALUES (?, ?, ?)", stadium_rows)

consecutive_ids_sql = """
WITH flagged AS (
  SELECT
    id, visit_date, people,
    LAG(people, 1) OVER (ORDER BY id) AS prev1_people,
    LAG(people, 2) OVER (ORDER BY id) AS prev2_people,
    LEAD(people, 1) OVER (ORDER BY id) AS next1_people,
    LEAD(people, 2) OVER (ORDER BY id) AS next2_people
  FROM Stadium
)
SELECT id, visit_date, people
FROM flagged
WHERE people >= 100
  AND (
    (prev1_people >= 100 AND prev2_people >= 100)
    OR (prev1_people >= 100 AND next1_people >= 100)
    OR (next1_people >= 100 AND next2_people >= 100)
  )
ORDER BY visit_date ASC;
"""

result = conn.execute(consecutive_ids_sql).fetchall()
print(result)
assert result == [(5, "2017-01-05", 145), (6, "2017-01-06", 1455), (7, "2017-01-07", 199), (8, "2017-01-09", 188)]
print("Stadium consecutive-ids query verified.")


## Part 2 -- SQL: Duplicate EmployeeIDs

**Idea:** `GROUP BY EmpID HAVING COUNT(*) > 1` -- the canonical "find duplicates" idiom. If full duplicate *rows* (not just the ID) were needed, `ROW_NUMBER() OVER (PARTITION BY EmpID ORDER BY EmpID)` filtered to `rn > 1` would return every row after the first for each duplicated ID.


In [ ]:
conn.execute("CREATE TABLE Employee (EmpID INTEGER, Name TEXT, Salary INTEGER, Department TEXT)")
conn.executemany("INSERT INTO Employee VALUES (?, ?, ?, ?)", [
    (101, "Prakash", 1200, "IT"),
    (102, "Jackie", 1100, "Sales"),
    (102, "Jackie", 1200, "Sales"),
])

dup_sql = "SELECT EmpID FROM Employee GROUP BY EmpID HAVING COUNT(*) > 1"
dup_result = conn.execute(dup_sql).fetchall()
assert dup_result == [(102,)]
print("Duplicate EmployeeIDs:", dup_result)


## Part 3 -- SQL: DAU / MAU

**Idea:** DAU is `COUNT(DISTINCT user_id)` grouped by day; MAU is the same, grouped by month. The only real decision is which date-truncation function your dialect provides -- PostgreSQL: `DATE_TRUNC('month', date)`; MySQL: `DATE_FORMAT(date, '%Y-%m-01')`; SQL Server: `DATEFROMPARTS(YEAR(date), MONTH(date), 1)`; SQLite (used to actually run this below): `strftime('%Y-%m', date)`.


In [ ]:
conn.execute("""
    CREATE TABLE events (date TEXT, user_id INTEGER, activity_type TEXT, ts TEXT)
""")
conn.executemany("INSERT INTO events VALUES (?, ?, ?, ?)", [
    ("2024-01-01", 1, "login", "2024-01-01T09:00"),
    ("2024-01-01", 2, "login", "2024-01-01T09:05"),
    ("2024-01-02", 1, "login", "2024-01-02T09:00"),
    ("2024-02-01", 1, "login", "2024-02-01T09:00"),
    ("2024-02-01", 3, "login", "2024-02-01T09:10"),
])

dau_sql = "SELECT date, COUNT(DISTINCT user_id) AS dau FROM events GROUP BY date ORDER BY date"
dau = conn.execute(dau_sql).fetchall()
assert dau == [("2024-01-01", 2), ("2024-01-02", 1), ("2024-02-01", 2)]

# SQLite equivalent of DATE_TRUNC('month', date) / DATE_FORMAT(date, '%Y-%m-01')
mau_sql = """
SELECT strftime('%Y-%m', date) AS month, COUNT(DISTINCT user_id) AS mau
FROM events GROUP BY month ORDER BY month
"""
mau = conn.execute(mau_sql).fetchall()
assert mau == [("2024-01", 2), ("2024-02", 2)]
print("DAU:", dau)
print("MAU:", mau)


## Part 4 -- Data Modeling: Star Schema

**Idea:** the flat file (`cust_id, cust_name, ..., sale_amt, sale_qty, product, product_family, date, ..., store_id, store_location`) is at the transaction-line-item grain. Split it into one **fact table** (`fact_sales`: the measures + foreign keys) and one **dimension table** per real-world entity (`dim_customer`, `dim_product`, `dim_store`, `dim_date`). All three example queries ("total sales per store", "distinct customers", "top product") are then a `GROUP BY` + aggregate over `fact_sales`, optionally joined to a dimension table for a human-readable label.

We build the schema and run all three example queries below to confirm the design actually supports them.


In [ ]:
conn.executescript("""
CREATE TABLE dim_customer (cust_id INTEGER PRIMARY KEY, cust_name TEXT, cust_address TEXT);
CREATE TABLE dim_product (product_id INTEGER PRIMARY KEY, product_name TEXT, product_family TEXT);
CREATE TABLE dim_store (store_id INTEGER PRIMARY KEY, store_location TEXT);
CREATE TABLE fact_sales (
    sale_id INTEGER PRIMARY KEY,
    cust_id INTEGER REFERENCES dim_customer(cust_id),
    product_id INTEGER REFERENCES dim_product(product_id),
    store_id INTEGER REFERENCES dim_store(store_id),
    sale_date TEXT,
    sale_amt REAL,
    sale_qty INTEGER
);
""")

conn.executemany("INSERT INTO dim_customer VALUES (?, ?, ?)", [(1, "Alice", "1 Main St"), (2, "Bob", "2 Oak St")])
conn.executemany("INSERT INTO dim_product VALUES (?, ?, ?)", [(10, "Widget", "Hardware"), (11, "Gadget", "Electronics")])
conn.executemany("INSERT INTO dim_store VALUES (?, ?)", [(100, "Downtown"), (101, "Uptown")])
conn.executemany("INSERT INTO fact_sales VALUES (?, ?, ?, ?, ?, ?, ?)", [
    (1, 1, 10, 100, "2024-01-01", 50.0, 2),
    (2, 2, 11, 100, "2024-01-02", 300.0, 1),
    (3, 1, 11, 101, "2024-01-03", 150.0, 1),
])

sales_per_store = conn.execute(
    "SELECT store_id, SUM(sale_amt) FROM fact_sales GROUP BY store_id ORDER BY store_id"
).fetchall()
assert sales_per_store == [(100, 350.0), (101, 150.0)]

distinct_customers = conn.execute("SELECT COUNT(DISTINCT cust_id) FROM fact_sales").fetchone()[0]
assert distinct_customers == 2

top_product = conn.execute("""
    SELECT product_id, SUM(sale_amt) AS total FROM fact_sales
    GROUP BY product_id ORDER BY total DESC LIMIT 1
""").fetchone()
assert top_product == (11, 450.0)   # Gadget: 300 + 150

print("Sales per store:", sales_per_store)
print("Distinct customers:", distinct_customers)
print("Top grossing product:", top_product)


## Part 5 -- Coding: Weekly Aggregation

**Idea:** the example output implies **ISO weeks (Monday start)**, not a rolling 7-day window from the first timestamp -- `'2019-01-08'` is kept separate from `'2019-01-01'`/`'2019-01-02'` even though it's exactly 7 days after the first entry, which only makes sense if the boundary is a fixed calendar week, not a sliding window. `date - timedelta(days=date.weekday())` gives the Monday of that date's week (`weekday()` returns 0 for Monday); group by that anchor, preserving the order each week first appears.

**Time complexity:** O(n) -- one pass, O(1) work per timestamp. **Space complexity:** O(n).


In [ ]:
from datetime import datetime, timedelta
from typing import List


def weekly_aggregation(ts: List[str]) -> List[List[str]]:
    dates = [datetime.strptime(t, "%Y-%m-%d") for t in ts]
    groups: dict = {}
    order: List[str] = []
    for d in dates:
        week_start = d - timedelta(days=d.weekday())   # Monday of this date's week
        key = week_start.strftime("%Y-%m-%d")
        if key not in groups:
            groups[key] = []
            order.append(key)
        groups[key].append(d.strftime("%Y-%m-%d"))
    return [groups[k] for k in order]


ts = ["2019-01-01", "2019-01-02", "2019-01-08", "2019-02-01", "2019-02-02", "2019-02-05"]
result = weekly_aggregation(ts)
assert result == [
    ["2019-01-01", "2019-01-02"],
    ["2019-01-08"],
    ["2019-02-01", "2019-02-02"],
    ["2019-02-05"],
]
print(result)


## Part 6 -- Coding: Character Frequency Counter

**Idea:** `collections.Counter` over a filtered generator (`c for c in s if 'a' <= c <= 'z'`) is the idiomatic one-liner; a manual `dict.get(c, 0) + 1` loop is the equally-correct explicit version if asked to avoid `Counter`.

**Time complexity:** O(n) -- one pass over the string. **Space complexity:** O(1) -- at most 26 keys regardless of input length.


In [ ]:
from collections import Counter


def char_frequency(s: str) -> dict:
    return dict(Counter(c for c in s if "a" <= c <= "z"))


def char_frequency_manual(s: str) -> dict:
    freq: dict = {}
    for c in s:
        if "a" <= c <= "z":
            freq[c] = freq.get(c, 0) + 1
    return freq


test_string = "hellowhatisyourname"
expected = {"h": 2, "e": 2, "l": 2, "o": 2, "w": 1, "a": 2, "t": 1,
            "i": 1, "s": 1, "y": 1, "u": 1, "r": 1, "n": 1, "m": 1}
assert char_frequency(test_string) == expected
assert char_frequency_manual(test_string) == expected

# Non-lowercase characters (uppercase, digits, punctuation) are correctly ignored
assert char_frequency("Hi! 123 hi") == {"i": 2, "h": 1}
print("Character frequency:", char_frequency(test_string))


## Discussion -- remaining follow-up directions

- **Return the start/end of each qualifying streak instead of individual rows (Part 1).** Assign a group id via `id - ROW_NUMBER() OVER (ORDER BY id)` restricted to qualifying rows -- consecutive qualifying ids get the same group id (their difference from a strictly-increasing row number is constant within a run), then `GROUP BY` that id and take `MIN`/`MAX` of `visit_date`.
- **DAU/MAU stickiness ratio as one query (Part 3).** Compute both in CTEs keyed by month (DAU averaged per month, or DAU on a representative day), join on month, and divide -- `dau_avg / mau` in the same query.
- **10 billion rows, <100ms queries (Parts 1/3/4).** Pre-aggregation (materialized views refreshed on a schedule), columnar storage, and partitioning by date are the standard levers -- window functions and `GROUP BY` over the *raw* fact table stop being fast at that scale regardless of indexing.
- **Timestamps spanning multiple timezones (Part 5).** Normalize every timestamp to UTC first, pick one timezone to define "week boundary" in, then truncate to date and compute the Monday anchor as before -- the anchor computation is unchanged, only the normalization step is new.
- **Unicode character frequency (Part 6).** `'a' <= c <= 'z'` is ASCII-only by construction; for Unicode, drop the range check (or replace it with `str.isalpha()`/a script-aware check) and let `Counter` key on `ord(c)` or the character itself -- a fixed 26-slot array stops being appropriate once the alphabet is unbounded.


## 🧩 Patterns Learned

- **Window functions replace self-joins for "compare to neighboring rows."** `LAG`/`LEAD` compute a neighbor's value in the same single pass the sort already requires, instead of the O(n^2)/O(n^3) blowup of joining a table to itself.
- **`GROUP BY … HAVING` filters on the aggregate, not the row.** `WHERE` can't reference `COUNT(*)` because grouping hasn't happened yet when `WHERE` runs -- that's exactly what `HAVING` is for.
- **Star schema: one fact table at the finest grain, one dimension table per real-world entity.** Design the schema around the *query patterns* you were shown, not the shape of the incoming flat file.
- **"What does the example output actually imply?" beats guessing at ambiguous specs.** The weekly-aggregation problem never states whether weeks are Monday-start or a rolling window -- the given example output is the actual specification once you reverse-engineer it.
- **Related problems:** consecutive-runs-in-SQL (LeetCode 601 Human Traffic of Stadium, near-identical), rolling/window aggregations generally (moving averages via window frames), any "flat file to normalized schema" data-modeling exercise.
- **Common pitfalls:** trying to filter on an aggregate with `WHERE` instead of `HAVING`; assuming a "weekly" or "monthly" bucket definition without checking it against a given example; forgetting that `DATE_TRUNC`/`DATE_FORMAT`/`DATEFROMPARTS` are dialect-specific and stating one without naming the dialect.
